In [1]:
# from google.colab import drive
# drive.mount("/content/gdrive", force_remount=True)

In [2]:
# import sys
import os
# sys.path.append('/content/gdrive/MyDrive/Colab Notebooks/Stat_S68/data/')
os.chdir('./stat_csv')
os.getcwd()

'/home/tako/Kasetsart/statistics/stat_csv'

In [3]:
# ! pip install gensim

In [4]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline

### Load and Split Data

In [5]:
data = pd.read_csv('yelp_labelled_edited.txt', sep="\t")
data.head()

,Text,Review
0,Wow... Loved this place.,1
1,Crust is not good.,0
2,Not tasty and the texture was just nasty.,0
3,Stopped by during the late May bank holiday of...,1
4,The selection on the menu was great and so wer...,1


In [6]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(data['Text'], data['Review'], test_size=0.2, random_state=42)

In [7]:
print(X_train.shape)
print(X_test.shape)

(800,)
(200,)


### Text Preprocessing

In [8]:
import nltk
custom_dir = '.devenv/state/venv/nltk_data' 
nltk.download('punkt_tab', download_dir=custom_dir)
nltk.download('stopwords', download_dir=custom_dir)
nltk.download('wordnet', download_dir=custom_dir)

[nltk_data] Downloading package punkt_tab to
[nltk_data]     .devenv/state/venv/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     .devenv/state/venv/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     .devenv/state/venv/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [9]:
def preprocess_text(text):
    import re, string
    from nltk.tokenize import word_tokenize 
    from nltk.corpus import stopwords 
    from nltk.stem import WordNetLemmatizer    
    
    word_tokens = word_tokenize(text)
    word_tokens = [word.lower() for word in word_tokens] 
    word_tokens = [re.sub(r'[^\w\s]', '', token) for token in word_tokens if re.sub(r'[^\w\s]', '', token)]
    corpus_stop_words = set(stopwords.words('english')) 
    word_tokens = [word for word in word_tokens if not word in corpus_stop_words] 
    lemmatizer = WordNetLemmatizer()
    word_tokens = [lemmatizer.lemmatize(word) for word in word_tokens]
    return word_tokens

In [10]:
processed_X_train = [" ".join(preprocess_text(word)) for word in X_train]
processed_X_train

['worst salmon sashimi',
 'excellent new restaurant experienced frenchman',
 'went lunch service slow',
 'think restaurant suffers trying hard enough',
 'lunch great experience',
 'got home see driest damn wing ever',
 'delicious',
 'brought fresh batch fry thinking yay something warm',
 'tragedy struck',
 'awful service',
 'great food service huge portion give military discount',
 'waiter nt helpful friendly rarely checked u',
 'could barely stomach meal nt complain business lunch',
 'price think place would much rather gone',
 'fried rice dry well',
 'truly unbelievably good glad went back',
 'eclectic selection',
 'based subpar service received effort show gratitude business wo nt going back',
 'chip sals amazing',
 'service quick even go order like like',
 'greedy corporation never see another dime',
 'also serve indian naan bread hummus spicy pine nut sauce world',
 'happened next pretty putting',
 'side greek salad greek dressing tasty pita hummus refreshing',
 'pizza selection g

In [11]:
processed_X_test = [" ".join(preprocess_text(word)) for word in X_test]
processed_X_test

['nt gone go',
 'try airport experience tasty food speedy friendly service',
 'restaurant clean family restaurant feel',
 'personally love hummus pita baklava falafel baba ganoush amazing eggplant',
 'come hungry leave happy stuffed',
 'great place highly recommend',
 'best luck rude noncustomer service focused new management',
 'reasonably priced also',
 'worst foodservice',
 'seriously solid breakfast',
 'service terrible though',
 '2 time bad customer service',
 'tried go lunch madhouse',
 'food nt good',
 'meat pretty dry sliced brisket pulled pork',
 'overall great experience',
 'went bachi burger friend recommendation disappointed',
 'ordered albondigas soup warm tasted like tomato soup frozen meatball',
 'place clean food oh stale',
 'bread made inhouse',
 'boyfriend came first time recent trip vega could pleased quality food service',
 'everything menu terrific also thrilled made amazing accommodation vegetarian daughter',
 'dropped ball',
 'sure order dessert even need pack to

###### Expected processed_train output = Nested list
###### sublist = list of tokenized words from each review
###### [['greek', 'dressing', 'creamy', 'flavorful'],
######  ['food', 'amazing'],
######  ['never', 'going', 'back'],
######  ['loved', 'place'],
######  ['came', 'back', 'today', 'since', 'relocated', 'still', 'impressed'],
######  ['liked', 'patio', 'service', 'outstanding'],..]


### Text Representation

In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(stop_words='english')

train_result = tfidf.fit_transform(processed_X_train)
test_result = tfidf.transform(processed_X_test)
print(tfidf.get_feature_names_out())
print(train_result.toarray())

['10' '100' '12' ... 'yum' 'yummy' 'zero']
[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]


In [13]:
feature_names = tfidf.get_feature_names_out()
tfidf_vectors = pd.DataFrame(train_result.toarray(), columns=feature_names)
print(tfidf_vectors)

      10  100   12      15lb   17  1979   20  2007   30     34ths  ...  wrong  \
0    0.0  0.0  0.0  0.000000  0.0   0.0  0.0   0.0  0.0  0.000000  ...    0.0   
1    0.0  0.0  0.0  0.000000  0.0   0.0  0.0   0.0  0.0  0.000000  ...    0.0   
2    0.0  0.0  0.0  0.000000  0.0   0.0  0.0   0.0  0.0  0.000000  ...    0.0   
3    0.0  0.0  0.0  0.000000  0.0   0.0  0.0   0.0  0.0  0.000000  ...    0.0   
4    0.0  0.0  0.0  0.000000  0.0   0.0  0.0   0.0  0.0  0.000000  ...    0.0   
..   ...  ...  ...       ...  ...   ...  ...   ...  ...       ...  ...    ...   
795  0.0  0.0  0.0  0.000000  0.0   0.0  0.0   0.0  0.0  0.000000  ...    0.0   
796  0.0  0.0  0.0  0.000000  0.0   0.0  0.0   0.0  0.0  0.000000  ...    0.0   
797  0.0  0.0  0.0  0.000000  0.0   0.0  0.0   0.0  0.0  0.000000  ...    0.0   
798  0.0  0.0  0.0  0.366272  0.0   0.0  0.0   0.0  0.0  0.366272  ...    0.0   
799  0.0  0.0  0.0  0.000000  0.0   0.0  0.0   0.0  0.0  0.000000  ...    0.0   

     yaall  yay  yeah  year

### Training Model and Evaluate Performance

In [14]:
# Classification model using X_train_vectors nd y_train as training dataset
# Use X_test_vectors as test dataset

In [15]:
def evaluate():
    y_predicted = clf.predict(test_result.toarray())
    # Evalute performance
    from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
    print(classification_report(y_test,y_predicted))
    
    print(y_predicted)
    
    print('Accuracy:', accuracy_score(y_test, y_predicted))
    print('Precision:', precision_score(y_test, y_predicted, pos_label=1))
    print('Recall:', recall_score(y_test, y_predicted, pos_label=1))
    print('F1 score:', f1_score(y_test, y_predicted, pos_label=1))
    
    print(confusion_matrix(y_test, y_predicted))

In [16]:
from sklearn.linear_model import LogisticRegression
clf = LogisticRegression(max_iter=1000)
clf.fit(tfidf_vectors, y_train)
evaluate()

              precision    recall  f1-score   support

           0       0.72      0.83      0.77        96
           1       0.82      0.70      0.76       104

    accuracy                           0.77       200
   macro avg       0.77      0.77      0.76       200
weighted avg       0.77      0.77      0.76       200

[0 1 1 1 1 1 0 1 0 1 0 0 0 0 0 1 0 0 1 1 1 1 0 0 0 0 0 0 1 0 0 1 0 1 0 1 1
 0 1 0 1 0 1 0 1 0 0 0 0 0 0 1 1 1 0 0 0 0 0 0 0 1 1 0 0 1 1 1 0 0 1 0 1 0
 1 0 1 0 0 0 0 1 0 1 0 0 0 1 1 1 0 0 0 1 0 1 1 0 1 0 1 0 0 1 1 1 1 0 0 0 0
 1 0 0 0 1 1 0 0 1 0 0 0 1 1 0 0 0 0 1 1 0 0 0 1 1 0 1 1 0 0 0 1 0 0 0 1 0
 1 0 0 1 0 1 0 1 1 0 0 0 0 1 0 0 1 1 1 1 1 1 1 1 1 0 0 0 1 0 0 0 1 0 1 0 1
 1 1 0 0 1 0 1 1 0 0 1 1 1 0 1]
Accuracy: 0.765
Precision: 0.8202247191011236
Recall: 0.7019230769230769
F1 score: 0.7564766839378239
[[80 16]
 [31 73]]


/nix/store/b5vbaqfa7nk850d8qxydkprx9x9vs0ca-devenv-profile/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(


In [17]:
### Try more classification models here

In [18]:
from sklearn.tree import DecisionTreeClassifier
clf = DecisionTreeClassifier(criterion = 'gini', random_state = 0)
clf.fit(tfidf_vectors, y_train)
evaluate()

              precision    recall  f1-score   support

           0       0.69      0.80      0.74        96
           1       0.78      0.66      0.72       104

    accuracy                           0.73       200
   macro avg       0.74      0.73      0.73       200
weighted avg       0.74      0.73      0.73       200

[0 1 1 1 1 1 1 1 0 0 0 0 1 0 0 1 0 0 0 1 1 1 0 0 0 0 1 0 1 0 1 1 0 0 1 1 1
 0 1 0 1 0 1 0 1 0 0 0 0 0 0 0 1 1 0 1 0 0 0 1 0 0 1 0 0 1 1 1 0 0 1 0 1 0
 1 1 1 0 0 0 1 1 0 1 0 0 0 1 1 1 1 0 0 0 0 1 1 0 1 0 1 0 0 1 0 1 1 0 1 1 1
 0 0 0 1 1 1 0 0 0 0 0 0 0 1 1 0 0 1 0 0 1 0 0 1 1 0 1 1 0 0 1 1 0 0 0 0 0
 1 0 0 1 0 1 0 0 0 0 0 0 0 1 0 0 1 0 0 1 1 1 1 1 1 0 0 1 1 0 0 0 0 0 1 0 1
 0 1 0 0 1 0 1 1 0 0 1 1 0 0 1]
Accuracy: 0.73
Precision: 0.7840909090909091
Recall: 0.6634615384615384
F1 score: 0.71875
[[77 19]
 [35 69]]


/nix/store/b5vbaqfa7nk850d8qxydkprx9x9vs0ca-devenv-profile/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but DecisionTreeClassifier was fitted with feature names
  warnings.warn(


In [19]:
selected_n = 100  # n_estimators  = number of decision trees used in random forest model, default value = 100
from sklearn.ensemble import RandomForestClassifier
clf = RandomForestClassifier(n_estimators = selected_n, criterion = 'gini', random_state = 0)
clf.fit(tfidf_vectors, y_train)
evaluate()

              precision    recall  f1-score   support

           0       0.64      0.89      0.75        96
           1       0.84      0.55      0.66       104

    accuracy                           0.71       200
   macro avg       0.74      0.72      0.70       200
weighted avg       0.74      0.71      0.70       200

[0 1 1 1 1 1 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 1 0 0 0 0 0 0 1 0 1 1 0 0 0 1 1
 0 1 0 1 0 1 0 1 0 0 0 0 0 0 1 1 1 0 0 0 0 0 0 0 0 1 0 0 0 1 1 0 0 0 0 1 0
 1 0 1 0 0 0 0 1 0 1 0 0 0 1 1 1 0 0 0 0 0 1 0 0 0 0 1 0 0 1 0 1 1 0 0 0 0
 0 0 0 0 1 1 0 0 1 0 0 0 0 1 0 0 0 0 0 1 0 0 0 1 1 0 1 1 0 0 0 1 0 0 0 0 0
 1 0 0 1 0 1 0 0 1 0 0 0 0 1 0 0 1 0 0 1 1 1 0 1 1 0 0 0 1 0 0 0 0 0 1 0 1
 1 1 0 0 1 0 1 1 0 0 1 1 1 0 1]
Accuracy: 0.71
Precision: 0.8382352941176471
Recall: 0.5480769230769231
F1 score: 0.6627906976744186
[[85 11]
 [47 57]]


/nix/store/b5vbaqfa7nk850d8qxydkprx9x9vs0ca-devenv-profile/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


In [20]:
# The model which perform the best of classification model of TF-IDF is the logistic regression